# Day 3: Embeddings and vector store

Study guide: `course/week-1/day-03-embeddings-vector-store/README.md`

Goal: embed all 1,405 chunks, store them in Postgres with pgvector, and search them by meaning.

**Setup:** reload edited `.py` files automatically. Run once after every kernel restart.

In [ ]:
%load_ext autoreload
%autoreload 2

## Step 1: Connect to the database

Postgres runs in Docker (`make up`). Python talks to it with `psycopg`.

In [ ]:
import psycopg

from research_rag.config import get_settings

settings = get_settings()
print("Connecting to:", settings.database_url)

with psycopg.connect(settings.database_url) as conn:
    row = conn.execute("SELECT version()").fetchone()
    print(row[0])

    distance = conn.execute("SELECT '[1,2,3]'::vector <=> '[1,2,4]'::vector").fetchone()[0]
    print("Cosine distance [1,2,3] vs [1,2,4]:", distance)

    distance = conn.execute("SELECT '[1,0]'::vector <=> '[0,1]'::vector").fetchone()[0]
    print("Cosine distance [1,0] vs [0,1]:   ", distance)

### Findings
- `<=>` is cosine **distance**: smaller = more similar. `distance = 1 − similarity`.
- `[1,2,3]` vs `[1,2,4]` ≈ 0.0085 (almost the same direction). `[1,0]` vs `[0,1]` = 1 (right angle, unrelated).

## Step 2: The table

`document_chunks` is defined in `db/init.sql` (8 columns, `embedding vector(384)`, HNSW index). `make reset` re-runs it. Check in the terminal:

```bash
docker exec -it research-rag-db psql -U rag -d research_rag -c "\\d document_chunks"
```

## Step 3: Embed chunks

`embed_texts()` in `src/research_rag/embeddings/embedder.py`: loads the model once (`@lru_cache`), embeds in batches of 64, normalizes to length 1.

In [ ]:
import numpy as np

from research_rag.embeddings.embedder import embed_texts
from research_rag.ingestion.models import Chunk
from research_rag.paths import DATA_DIR

# Load the 1,405 chunks from Day 2
chunks = []
with open(DATA_DIR / "processed" / "chunks.jsonl") as file:
    for line in file:
        chunk = Chunk.model_validate_json(line)   # JSON text -> checked Chunk object
        chunks.append(chunk)
print("Chunks loaded:", len(chunks))

# Embed the first 3
texts = []
for chunk in chunks[:3]:
    texts.append(chunk.text)

vectors = embed_texts(texts)
print("Shape:", vectors.shape)
print("Length of each vector:", np.linalg.norm(vectors, axis=1))
print("First 5 numbers of chunk 0:", vectors[0][:5])
print()
print("Similarity chunk 0 vs 1:", round(float(vectors[0] @ vectors[1]), 3))
print("Similarity chunk 0 vs 2:", round(float(vectors[0] @ vectors[2]), 3))

## Step 3: Embed chunks
_Filled in as we go._

In [3]:
import json

import numpy as np

from research_rag.embeddings.embedder import embed_texts
from research_rag.ingestion.models import Chunk
from research_rag.paths import DATA_DIR

# Load the 1,405 chunks from Day 2
chunks = []
with open(DATA_DIR / "processed" / "chunks.jsonl") as file:
    for line in file:
        chunks.append(Chunk.model_validate_json(line))
print("Chunks loaded:", len(chunks))

# Embed the first 3
texts = []
for chunk in chunks[:3]:
    texts.append(chunk.text)

vectors = embed_texts(texts)
print("Shape:", vectors.shape)
print("Length of each vector:", np.linalg.norm(vectors, axis=1))
print("First 5 numbers of chunk 0:", vectors[0][:5])
print()
print("Similarity chunk 0 vs 1:", round(float(vectors[0] @ vectors[1]), 3))
print("Similarity chunk 0 vs 2:", round(float(vectors[0] @ vectors[2]), 3))

/Users/ashutoshraichurkar/Documents/repos/research-rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1119 23:12:40.482000 22729 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Chunks loaded: 1405


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10387.43it/s]


Shape: (3, 384)
Length of each vector: [1. 1. 1.]
First 5 numbers of chunk 0: [-0.09647237 -0.13539645  0.04650646 -0.01371389  0.04165177]

Similarity chunk 0 vs 1: 0.531
Similarity chunk 0 vs 2: 0.503


## Step 4: Store chunks
_Filled in as we go._

In [4]:
from research_rag.embeddings.store import connect, count_chunks, upsert_chunks

with connect() as conn:
    print("Before:", count_chunks(conn))

    upsert_chunks(conn, chunks[:3], vectors)
    print("After 1st run:", count_chunks(conn))

    upsert_chunks(conn, chunks[:3], vectors)   # exactly the same chunks again
    print("After 2nd run:", count_chunks(conn))

Before: 0
After 1st run: 3
After 2nd run: 3


## Step 5: Load all 1,405 chunks
_Filled in as we go._

## Step 6: First vector search
_Filled in as we go._

In [5]:
from research_rag.embeddings.embedder import embed_texts
from research_rag.embeddings.store import connect

SEARCH_SQL = """
SELECT id, source, page_number, text,
       1 - (embedding <=> %s) AS similarity
FROM document_chunks
ORDER BY embedding <=> %s
LIMIT %s
"""


def search(question: str, k: int = 5) -> list[tuple]:
    """Embed the question and return the k most similar chunks."""
    question_vector = embed_texts([question])[0]      # one question -> one row of 384 numbers
    with connect() as conn:
        rows = conn.execute(SEARCH_SQL, (question_vector, question_vector, k)).fetchall()
    return rows


def show(question: str, k: int = 5) -> None:
    """Print the search results in a readable way."""
    print("Q:", question)
    for chunk_id, source, page, text, similarity in search(question, k):
        print(f"  {similarity:.3f}  {chunk_id:<24} p.{page:<3} {text[:90]}...")
    print()


show("Does LoRA add inference latency?")

Q: Does LoRA add inference latency?
  0.648  2106.09685.pdf::0021     p.4   LoRA roughly converges to training the original model, while adapter-based methods converg...
  0.586  2305.14314.pdf::0030     p.6   under what circumstances slowdowns occur from the paging process. Default LoRA hyperparame...
  0.584  2106.09685.pdf::0020     p.4   a result, we simply set α to the first r we try and do not tune it. This scaling helps to ...
  0.564  2106.09685.pdf::0024     p.5   350GB to 35MB)4. This allows us to train with significantly fewer GPUs and avoid I/O bottl...
  0.546  2305.14314.pdf::0124     p.22  65B). We find LoRA r is unrelated to final performance if LoRA is used on all layers as ca...



In [6]:
show("How does QLoRA reduce memory when fine-tuning?")            # should find QLoRA (2305.14314)
show("Why do models struggle with information in the middle of long contexts?")  # Lost in the Middle (2307.03172)
show("What is NF4?")                                                # an acronym: the weak spot
show("What's a good recipe for banana bread?")                     # nothing in your papers covers this

Q: How does QLoRA reduce memory when fine-tuning?
  0.669  2305.14314.pdf::0024     p.5   works like regular memory paging between CPU RAM and the disk. We use this feature to allo...
  0.628  2305.14314.pdf::0143     p.26  length 512 and is estimated only for adapters and the base model weights (no attention). N...
  0.603  2305.14314.pdf::0037     p.7   memory GPUs, we continue to test whether 4-bit QLORA can match 16-bit LoRA at the 7B to 65...
  0.589  2305.14314.pdf::0000     p.1   QLORA: Efficient Finetuning of Quantized LLMs Tim Dettmers∗ Artidoro Pagnoni∗ Ari Holtzman...
  0.573  2305.14314.pdf::0026     p.5   the derivative ∂X ∂W in BFloat16 precision. To summarize, QLORA has one storage data type ...

Q: Why do models struggle with information in the middle of long contexts?
  0.572  2307.03172.pdf::0000     p.1   Lost in the Middle: How Language Models Use Long Contexts Nelson F. Liu1∗ Kevin Lin2 John ...
  0.563  2307.03172.pdf::0056     p.10  we observe in this work has a 

### Findings
- QLoRA memory question: all 5 hits from QLoRA, top 0.669 (paged memory). Clean semantic match.
- Lost in the Middle: right paper, but title chunk and 2 references-section chunks in the top 5 (noise).
- "What is NF4?": right paper but weak (top 0.426); the chunk that mentions NF4 ranked 4th. Acronym weakness → Day 8 hybrid search.
- Banana bread (no answer exists): still returned 5 results, top 0.482 (ReAct shopping examples mention "banana").
  It scored HIGHER than the valid NF4 question, so one similarity threshold can't separate relevant from irrelevant.